# Pipeline walkthrough — every processing step, visualized in order

Follow one band through the full reverse-E2ES chain and **see what each step does to the image**:

```
L1A DN ──forward_radiometric──▶ radiance                      (the processor's view)
radiance ─S1▶ signal DN ─S6▶ +PSF blur ─S7▶ +PRNU ─S13▶ +noise
         ─S11▶ +dark ─S12▶ +onboard-eq ─S14▶ 12-bit RAW DN     (ATBD §5 reverse chain)
RAW ─S15/package▶ CCSDS-122 ISPs ─ground-decode▶ DN′ == RAW    (bit-exact, SentiWiki L0→L1A)
     └─ the same steps the `package` / `ground-decode` / `l0-decode` pipeline phases run
```

- Input: a window of the **real L1A** from the store (`$S2_DATA_STORE`, default `~/data-store`);
  falls back to a synthetic scene when no store is present.
- The ADF here is the seeded synthetic set (`adf.synthesize`) unless `S2_E2ES_GIPP_DIR` is set —
  then the real per-pixel GIPP coefficients load and the derived radiance is operational.
- Tip: if saving fails with a 413 (large embedded figures), `Edit ▸ Clear Outputs of All Cells` then save.

In [ ]:
# One-time kernel setup — safe to re-run. After a first-time install, RESTART the kernel.
try:
    import s2_msi_raw_generator  # noqa: F401
    print("s2_msi_raw_generator OK")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install --quiet -e .. --no-deps
    !{sys.executable} -m pip install --quiet matplotlib zarr
    print("installed — now RESTART the kernel (Kernel ▸ Restart Kernel…) and run all cells again")

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from s2_msi_raw_generator import adf as adfmod, ccsds122, io as gio, isp, l0product, reverse, sensor

plt.rcParams["figure.dpi"] = 80

STORE = Path(os.environ.get("S2_DATA_STORE") or "~/data-store").expanduser()
BAND, DET, LINES = "B04", 1, 512  # one 10 m band, first detector, a 512-line window


def stretch(img, max_lines=None):
    im = np.asarray(img if max_lines is None else img[:max_lines], dtype=np.float64)
    lo, hi = np.percentile(im, (2, 98))
    return np.clip(im, lo, min(hi, lo + max(hi - lo, 1e-9)))


def show(img, title, cmap="gray"):
    plt.figure(figsize=(9, 4))
    plt.imshow(stretch(img), cmap=cmap, aspect="auto", interpolation="nearest")
    plt.title(title); plt.xlabel("detector column"); plt.ylabel("line")
    plt.colorbar(); plt.tight_layout(); plt.show()


def panel(stages, ncols=3):
    """Grid of (name, description, image) triples — the chain at a glance."""
    rows = -(-len(stages) // ncols)
    fig, axes = plt.subplots(rows, ncols, figsize=(14, 3.2 * rows))
    for ax, (name, desc, img) in zip(np.ravel(axes), stages):
        ax.imshow(stretch(img), cmap="gray", aspect="auto", interpolation="nearest")
        ax.set_title(f"{name} — {desc}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in np.ravel(axes)[len(stages):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

## Step 0 — the input: real L1A raw DN

Raw instrument counts as downlinked (dark pedestal + PRNU still imprinted). This is what the
`preflight`/`package` phases read via `io.read_l1a_raw`.

In [ ]:
l1a_p = STORE / "inputs" / "PDI_MSI_S2_L1A.zarr"
if l1a_p.exists():
    dn_in = gio.read_l1a_raw(str(l1a_p), DET, BAND, lines=slice(0, LINES), dtype=np.float64)
    src = f"real L1A ({l1a_p.name})"
else:  # synthetic fallback so the walkthrough also runs without a store
    rng0 = np.random.default_rng(11)
    y, x = np.mgrid[0:LINES, 0:2592].astype(np.float64)
    dn_in = np.clip(900 + 600 * np.sin(y / 90) * np.cos(x / 240)
                    + 40 * rng0.standard_normal((LINES, 2592)), 0, 4095)
    src = "synthetic scene (no store found)"

WIDTH = dn_in.shape[1]
print(f"input: {src} — {BAND} d{DET:02d}, window {dn_in.shape}, DN [{dn_in.min():.0f}, {dn_in.max():.0f}]")
show(dn_in, f"Step 0 — input L1A raw DN ({BAND}, {src})")
plt.figure(figsize=(7, 2.5)); plt.hist(dn_in.ravel(), bins=100)
plt.title("input DN histogram"); plt.xlabel("DN"); plt.ylabel("px"); plt.tight_layout(); plt.show()

## Bridge — DN → radiance (`forward_radiometric`)

The processor's algebraic forward model (undo S12 → S11 → S7 → S1) turns raw DN into radiance —
the physical quantity the reverse chain will then degrade back to RAW, step by step.

In [ ]:
b = sensor.band(BAND)
gipp_dir = os.environ.get("S2_E2ES_GIPP_DIR")
if gipp_dir:
    from s2_msi_raw_generator import gipp as gipp_mod
    A = adfmod.BandADF.from_gipp(b, DET, gipp_mod.load_gipp_set(gipp_dir), active_width=WIDTH)
    adf_kind = "real operational GIPP (per-pixel dark + relative response)"
else:
    A = adfmod.synthesize(b, n_det=WIDTH, seed=7)
    adf_kind = "synthetic seeded ADF (set S2_E2ES_GIPP_DIR for the real GIPP)"
print(f"ADF: {adf_kind}\ncal_gain A = {A.band.cal_gain:.3f}  noise α,β = {A.noise_a:.3f}, {A.noise_b:.4f}")

radiance = reverse.forward_radiometric(dn_in, A)
show(radiance, "Bridge — derived radiance L = fwd(DN)  [W·m⁻²·sr⁻¹·µm⁻¹]")

## The reverse chain, step by step (S1 → S14)

Same order as `reverse.reverse_mvp` — noise (S13) is impressed on the *signal* DN before the
dark pedestal (S11), so σ = √(α² + β·DN) reproduces the product SNR@Lref exactly.

In [ ]:
rng = np.random.default_rng(2026)
stages = [("Step 0", "input L1A DN", dn_in), ("Bridge", "radiance (processor view)", radiance)]

s = reverse.s1_radiance_to_dn(radiance, A.band.cal_gain)
stages.append(("S1", "radiance → signal DN (·cal_gain)", s))
s = reverse.s6_psf_reblur(s, A.psf)
stages.append(("S6", "optical PSF re-blur (real ESA kernel)", s))
s = reverse.s7_impress_relative_response(s, A.prnu_gain)
stages.append(("S7", "impress per-detector PRNU (÷gain)", s))
s = reverse.s13_add_noise(s, A.noise_a, A.noise_b, rng)
stages.append(("S13", "add sensor noise σ=√(α²+β·DN)", s))
s = reverse.s11_reapply_dark(s, A.dark_dn)
stages.append(("S11", "re-add dark pedestal (+D per column)", s))
s = reverse.s12_reapply_onboard_eq(s, A.eq_gain, A.eq_offset)
stages.append(("S12", "reverse onboard equalization", s))
raw = reverse.s14_quantize(s)
stages.append(("S14", "quantize+clip → 12-bit RAW uint16", raw))

panel(stages)
print(f"RAW DN range [{raw.min()}, {raw.max()}]  (12-bit ceiling {sensor.DN_MAX})")

### What each step actually changed

Left: the coefficients the steps applied (PSF kernel, per-column PRNU/dark/eq profiles).
Right: image-space differences — the blur field (S6), the impressed noise (S13) and the final
12-bit histogram (S14).

In [ ]:
arr = {name: img for name, _, img in stages}
fig, ax = plt.subplots(2, 3, figsize=(14, 6))
ax[0, 0].imshow(A.psf, cmap="viridis"); ax[0, 0].set_title("S6 — PSF kernel (DC-unit)", fontsize=9)
ax[0, 1].plot(A.prnu_gain, lw=0.7); ax[0, 1].set_title("S7 — PRNU gain per column", fontsize=9)
ax[0, 2].plot(A.dark_dn, lw=0.7); ax[0, 2].plot(A.eq_gain, lw=0.7)
ax[0, 2].set_title("S11 dark DN / S12 eq gain per column", fontsize=9)
ax[0, 2].legend(["dark_dn", "eq_gain"], fontsize=8)
d_blur = np.asarray(arr["S6"]) - np.asarray(arr["S1"])
ax[1, 0].imshow(stretch(d_blur), cmap="coolwarm", aspect="auto")
ax[1, 0].set_title("S6 − S1: blur field", fontsize=9)
d_noise = np.asarray(arr["S13"]) - np.asarray(arr["S7"])
ax[1, 1].imshow(stretch(d_noise), cmap="coolwarm", aspect="auto")
ax[1, 1].set_title(f"S13 − S7: impressed noise (σ̂={d_noise.std():.2f} DN)", fontsize=9)
ax[1, 2].hist(np.asarray(arr["S14"]).ravel(), bins=100)
ax[1, 2].set_title("S14 — RAW 12-bit DN histogram", fontsize=9)
for a in ax[0]:
    a.tick_params(labelsize=7)
for a in (ax[1, 0], ax[1, 1]):
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## S15 / `package` phase — CCSDS-122 compression + space packets

The RAW frame is losslessly compressed (integer 9/7-M DWT + bit-plane coder) and carried as
real CCSDS space packets. First: the 3-level DWT subband decomposition the codec starts from —
energy concentrates in LL3, the detail subbands are sparse (that sparsity IS the compression).

In [ ]:
crop = raw[: (LINES // 8) * 8, : (WIDTH // 8) * 8]
sub = ccsds122.dwt97m_forward(crop)
names = ["LL3", "HL1", "LH1", "HH1"]
fig, ax = plt.subplots(1, 4, figsize=(14, 2.8))
for a, n in zip(ax, names):
    a.imshow(stretch(np.abs(sub[n])), cmap="magma", aspect="auto", interpolation="nearest")
    a.set_title(f"{n}  {sub[n].shape}", fontsize=9); a.set_xticks([]); a.set_yticks([])
plt.suptitle("CCSDS-122 integer 9/7-M DWT subbands (|coefficients|)", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
payload, stats = ccsds122.compress_frame(raw, pixel_bit_depth=12)
bounds = ccsds122.segment_byte_bounds(payload)
apid = isp.apid_for(DET, sensor.BANDS.index(BAND))
seg_times = 1e9 + np.arange(len(bounds)) * (8 * sensor.LINE_PERIOD_MS / 1e3)
stream, offsets, plens = isp.packetize_stream(
    payload, apid, segment_bounds=bounds, segment_times_gps=seg_times
)
print(f"compressed: {stats.raw_bytes/1e6:.2f} MB → {stats.compressed_bytes/1e6:.2f} MB  "
      f"(ratio ×{stats.ratio:.2f}, {stats.n_segments} segments of 8 lines)")
print(f"packetized: {offsets.size} CCSDS space packets, APID {apid}, stream {stream.size/1e6:.2f} MB")

seg_sizes = np.diff(np.concatenate([[0], np.asarray(bounds, dtype=np.int64)]))
fig, ax = plt.subplots(1, 2, figsize=(12, 2.8))
ax[0].plot(seg_sizes, lw=0.7)
ax[0].set_title("codec segment size per 8-line block row [bytes]", fontsize=9)
ax[0].set_xlabel("segment #", fontsize=8)
ax[1].hist(plens, bins=40)
ax[1].set_title("space-packet data length distribution [bytes]", fontsize=9)
plt.tight_layout(); plt.show()

## `ground-decode` phase — ISPs back to DN, bit-exact

The L1A-side operation (SentiWiki: decompression happens at L1A). The packets are reassembled
(sequence flags + continuity enforced) and the CCSDS-122 stream decoded — the result must equal
the RAW frame **bit for bit**, which is exactly what the pipeline asserts for all 13 bands.

In [ ]:
rejoined = b"".join(isp.reassemble_segments(stream))
dn_back = ccsds122.decompress_frame(rejoined)
exact = bool(np.array_equal(dn_back, raw))
print(f"decoded {dn_back.shape} {dn_back.dtype} — bit-exact vs RAW: {exact}")
assert exact, "codec round-trip must be lossless"
diff = dn_back.astype(np.int32) - raw.astype(np.int32)
show(diff, f"ground-decode − RAW  (max |Δ| = {np.abs(diff).max()})", cmap="coolwarm")

## The same steps, as the real pipeline ran them (from the store)

The `package` → `ground-decode` → `l0-decode` phases persisted their results — compare the
walkthrough against the actual products and phase reports.

In [ ]:
import zarr

canon = sorted(p for p in (STORE / "l0").glob("*.zarr") if "_OC" not in p.name) if (STORE / "l0").is_dir() else []
if canon:
    g = zarr.open_group(str(canon[0]), mode="r")
    bg = g[f"measurements/d{DET:02d}/{sensor.zarr_band_key(BAND)}"]
    stored = np.asarray(bg[f"band{sensor.band_number(BAND)}"][:LINES])
    show(stored, f"store: canonical L0 decoded DN — {canon[0].name} ({BAND})")
    c = dict(bg.attrs)["compression"]
    print(f"pipeline compression for {BAND}: ratio ×{c['ratio']} "
          f"({c['raw_bytes']/1e6:.1f} → {c['compressed_bytes']/1e6:.1f} MB)  vs walkthrough ×{stats.ratio:.2f}")
else:
    print("no canonical L0 in the store — run: python scripts/run_pipeline.py")

gd_json = STORE / "report" / "ground_decode.json"
if gd_json.exists():
    data = json.loads(gd_json.read_text())
    print(f"\nground-decode phase: {'band':6s} {'bit_exact':>9s} {'ratio':>6s} {'packets':>8s}")
    for bn, r in data.items():
        print(f"                     {bn:6s} {str(r.get('bit_exact')):>9s} {r.get('ratio', 0):6.2f} {r.get('n_packets', 0):8d}")

prime = sorted((STORE / "l1a_prime").glob("*.zarr")) if (STORE / "l1a_prime").is_dir() else []
for gp in prime:
    gg = zarr.open_group(str(gp), mode="r")
    try:
        det_grp = gg["measurements/detector"]
    except KeyError:
        continue
    if BAND in dict(det_grp.arrays()):
        prime_dn = np.asarray(det_grp[BAND][:LINES])
        show(prime_dn, f"store: L1A′ after msi-processor l0-decode — {gp.name} ({BAND})")
        if l1a_p.exists():
            orig = gio.read_l1a_raw(str(l1a_p), DET, BAND, lines=slice(0, min(LINES, prime_dn.shape[0])), dtype=np.float64)
            d = prime_dn[: orig.shape[0]].astype(np.float64) - orig
            print(f"L1A′ vs original L1A on the window: max|Δ| = {np.abs(d).max():.1f} DN")
        break
else:
    print("no L1A' products — run the l0-decode phase (needs eopf + msi_processor)")

---
**Where to go deeper:** `docs/atbd/atbd.md` (each S-step's theory), `docs/dpm/` (block diagrams),
`notebooks/inspect_products.ipynb` (full store browser: inventory, metadata, quicklooks, reports),
`S2_E2ES_PHASES=figures S2_E2ES_L1B=<L1B> python scripts/run_pipeline.py` (the committed showcase
figures with a real L1B and quality-metric tables).